# 🔍 NodeWorks Knowledge Graph — Data Exploration

This notebook reads live data from the Neo4j Knowledge Graph and explores:
- Node counts per label
- Relationship counts per type
- Per-freelancer breakdown (skills, domains, roles, experience)
- Most common skills and domains across all freelancers
- Role distribution and top-scoring candidates
- Freelancer similarity (shared skills / shared nodes)
- Graph density and connectivity stats

## 0 · Setup & Connection

In [ ]:
# ── Install dependencies if needed ─────────────────────────────────────────
import subprocess, sys
for pkg in ["neo4j", "python-dotenv", "pandas", "matplotlib", "seaborn"]:
    try:
        __import__(pkg.replace("-", "_").split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load .env from project root (adjust path if needed)
load_dotenv(Path("MergedCVAnalyzer-with-KBS/.env"))

# ── Plot style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
ACCENT = "#4C72B0"

print("✅ Libraries loaded")

In [ ]:
# ── Connect to Neo4j ────────────────────────────────────────────────────────
URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USER     = os.getenv("NEO4J_USER",     "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {URI}")

def run(query: str, **params) -> list[dict]:
    """Run a Cypher query and return results as a list of dicts."""
    with driver.session() as s:
        return [r.data() for r in s.run(query, **params)]

---
## 1 · Graph Overview — Node & Relationship Counts

In [ ]:
# ── Node counts per label ───────────────────────────────────────────────────
node_counts = run("""
    CALL db.labels() YIELD label
    CALL apoc.cypher.run('MATCH (n:' + label + ') RETURN count(n) AS cnt', {})
    YIELD value
    RETURN label, value.cnt AS count
    ORDER BY count DESC
""")

# Fallback without APOC
if not node_counts:
    labels = ["Freelancer", "Skill", "Domain", "Company", "Institution", "Project", "Role"]
    node_counts = []
    for lbl in labels:
        res = run(f"MATCH (n:{lbl}) RETURN count(n) AS count")
        node_counts.append({"label": lbl, "count": res[0]["count"]})

df_nodes = pd.DataFrame(node_counts).set_index("label").sort_values("count", ascending=False)
print("=" * 40)
print("  NODE COUNTS")
print("=" * 40)
print(df_nodes.to_string())
print(f"\n  TOTAL NODES : {df_nodes['count'].sum():,}")

In [ ]:
# ── Relationship counts per type ────────────────────────────────────────────
rel_types = [
    "HAS_SKILL", "SKILL_OWNED_BY",
    "HAS_DOMAIN", "DOMAIN_OF",
    "WORKED_AT", "EMPLOYED",
    "STUDIED_AT", "ALUMNI_OF",
    "CREATED_PROJECT", "DEVELOPED_BY",
    "USED_TECH", "USED_IN",
    "MATCHES_ROLE", "SUITABLE_CANDIDATE",
]

rel_counts = []
for rt in rel_types:
    res = run(f"MATCH ()-[r:{rt}]->() RETURN count(r) AS count")
    rel_counts.append({"type": rt, "count": res[0]["count"]})

df_rels = pd.DataFrame(rel_counts).set_index("type").sort_values("count", ascending=False)
print("=" * 40)
print("  RELATIONSHIP COUNTS")
print("=" * 40)
print(df_rels.to_string())
print(f"\n  TOTAL RELATIONSHIPS : {df_rels['count'].sum():,}")

In [ ]:
# ── Bar charts: nodes and relationships side by side ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_nodes["count"].plot(
    kind="barh", ax=axes[0], color=ACCENT, edgecolor="white"
)
axes[0].set_title("Node Counts by Label", fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].invert_yaxis()
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)

df_rels["count"].plot(
    kind="barh", ax=axes[1], color="#DD8452", edgecolor="white"
)
axes[1].set_title("Relationship Counts by Type", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width())}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

---
## 2 · Freelancer Profiles

In [ ]:
# ── Pull full per-freelancer summary ────────────────────────────────────────
rows = run("""
    MATCH (f:Freelancer)
    OPTIONAL MATCH (f)-[:HAS_SKILL]->(s:Skill)
    WITH f, count(DISTINCT s) AS n_skills
    OPTIONAL MATCH (f)-[:HAS_DOMAIN]->(d:Domain)
    WITH f, n_skills, count(DISTINCT d) AS n_domains
    OPTIONAL MATCH (f)-[:WORKED_AT]->(c:Company)
    WITH f, n_skills, n_domains, count(DISTINCT c) AS n_companies
    OPTIONAL MATCH (f)-[:STUDIED_AT]->(i:Institution)
    WITH f, n_skills, n_domains, n_companies, count(DISTINCT i) AS n_institutions
    OPTIONAL MATCH (f)-[:CREATED_PROJECT]->(p:Project)
    WITH f, n_skills, n_domains, n_companies, n_institutions, count(DISTINCT p) AS n_projects
    OPTIONAL MATCH (f)-[mr:MATCHES_ROLE]->(r:Role)
    WITH f, n_skills, n_domains, n_companies, n_institutions, n_projects,
         count(mr) AS n_roles,
         max(mr.score) AS best_score
    OPTIONAL MATCH (f)-[top_mr:MATCHES_ROLE]->(best_role:Role)
    WHERE top_mr.score = best_score
    RETURN
        f.name                AS name,
        f.email               AS email,
        f.years_of_experience AS years_exp,
        n_skills, n_domains, n_companies,
        n_institutions, n_projects, n_roles,
        round(best_score, 1)  AS best_score,
        best_role.name        AS best_role
    ORDER BY best_score DESC
""")

df_fl = pd.DataFrame(rows)
df_fl["years_exp"] = df_fl["years_exp"].fillna(0.0).round(2)
print(f"👥 {len(df_fl)} freelancers in the graph\n")
df_fl

In [ ]:
# ── Summary statistics ──────────────────────────────────────────────────────
cols = ["n_skills", "n_domains", "n_companies", "n_institutions", "n_projects", "years_exp", "best_score"]
print("📊 Descriptive Statistics")
df_fl[cols].describe().round(2)

In [ ]:
# ── Per-freelancer skill & domain breakdown bar chart ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_plot = df_fl.set_index("name").sort_values("n_skills", ascending=True)

df_plot["n_skills"].plot(kind="barh", ax=axes[0], color=ACCENT, edgecolor="white")
axes[0].set_title("Skills per Freelancer", fontweight="bold")
axes[0].set_xlabel("# Skills")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width())}", va="center", fontsize=8)

df_plot2 = df_fl.set_index("name").sort_values("n_domains", ascending=True)
df_plot2["n_domains"].plot(kind="barh", ax=axes[1], color="#55A868", edgecolor="white")
axes[1].set_title("Domain Knowledge per Freelancer", fontweight="bold")
axes[1].set_xlabel("# Domains")
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width())}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Experience distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_fl.sort_values("years_exp", ascending=True).set_index("name")["years_exp"].plot(
    kind="barh", ax=axes[0], color="#C44E52", edgecolor="white"
)
axes[0].set_title("Years of Experience per Freelancer", fontweight="bold")
axes[0].set_xlabel("Years")
axes[0].axvline(df_fl["years_exp"].mean(), color="black", linestyle="--", linewidth=1, label="Mean")
axes[0].legend()
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{bar.get_width():.1f}", va="center", fontsize=8)

axes[1].hist(df_fl["years_exp"], bins=8, color="#C44E52", edgecolor="white", rwidth=0.85)
axes[1].set_title("Experience Distribution", fontweight="bold")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("# Freelancers")
axes[1].axvline(df_fl["years_exp"].mean(), color="black", linestyle="--", linewidth=1, label=f"Mean {df_fl['years_exp'].mean():.1f}yr")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3 · Skills Analysis

In [ ]:
# ── Most common skills across all freelancers ───────────────────────────────
skill_rows = run("""
    MATCH (f:Freelancer)-[:HAS_SKILL]->(s:Skill)
    RETURN s.name AS skill, count(DISTINCT f) AS freelancer_count
    ORDER BY freelancer_count DESC
""")

df_skills = pd.DataFrame(skill_rows)
print(f"🛠  Total unique skills in the graph: {len(df_skills)}")
print("\nTop 20 most common skills:")
df_skills.head(20)

In [ ]:
# ── Top-20 skills bar chart ─────────────────────────────────────────────────
top20 = df_skills.head(20).sort_values("freelancer_count", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20["skill"], top20["freelancer_count"], color=ACCENT, edgecolor="white")
ax.set_title("Top 20 Most Common Skills", fontweight="bold", fontsize=13)
ax.set_xlabel("Number of Freelancers with this Skill")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Rare skills (held by only 1 freelancer) ─────────────────────────────────
rare = df_skills[df_skills["freelancer_count"] == 1]
print(f"🔬 Rare skills (unique to a single freelancer): {len(rare)}")
print(rare["skill"].tolist())

In [ ]:
# ── Skills distribution: how many freelancers own N skills ──────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_fl["n_skills"], bins=12, color=ACCENT, edgecolor="white", rwidth=0.85)
ax.set_title("Distribution of Skill Count per Freelancer", fontweight="bold")
ax.set_xlabel("Number of Skills")
ax.set_ylabel("Number of Freelancers")
ax.axvline(df_fl["n_skills"].mean(), color="#C44E52", linestyle="--",
           label=f"Mean: {df_fl['n_skills'].mean():.1f}")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4 · Domain Knowledge Analysis

In [ ]:
# ── Most common domains ─────────────────────────────────────────────────────
domain_rows = run("""
    MATCH (f:Freelancer)-[:HAS_DOMAIN]->(d:Domain)
    RETURN d.name AS domain, count(DISTINCT f) AS freelancer_count
    ORDER BY freelancer_count DESC
""")

df_domains = pd.DataFrame(domain_rows)
print(f"🌐 Total unique domains in the graph: {len(df_domains)}")
df_domains.head(20)

In [ ]:
# ── Top-20 domains chart ────────────────────────────────────────────────────
top20_d = df_domains.head(20).sort_values("freelancer_count", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20_d["domain"], top20_d["freelancer_count"], color="#55A868", edgecolor="white")
ax.set_title("Top 20 Most Common Domains", fontweight="bold", fontsize=13)
ax.set_xlabel("Number of Freelancers with this Domain")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5 · Role Distribution & Best-Match Scores

In [ ]:
# ── Which role each freelancer best fits ────────────────────────────────────
best_role_counts = df_fl["best_role"].value_counts()
print("🏆 Best-fit role distribution:")
print(best_role_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
best_role_counts.sort_values().plot(kind="barh", ax=ax, color="#8172B2", edgecolor="white")
ax.set_title("Best-Fit Role Distribution Across Freelancers", fontweight="bold")
ax.set_xlabel("Number of Freelancers")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar in ax.patches:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top candidates per role (highest skill score) ───────────────────────────
role_top = run("""
    MATCH (f:Freelancer)-[mr:MATCHES_ROLE]->(r:Role)
    WITH r.name AS role, f.name AS freelancer, mr.score AS score
    ORDER BY role, score DESC
    WITH role, collect({name: freelancer, score: score})[0] AS top
    RETURN role, top.name AS top_candidate, round(top.score, 1) AS top_score
    ORDER BY top_score DESC
    LIMIT 20
""")

df_role_top = pd.DataFrame(role_top)
print("Top candidate per role (top 20 by score):")
df_role_top

In [ ]:
# ── Score distribution across all freelancers ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_fl["best_score"].dropna(), bins=12, color="#8172B2", edgecolor="white", rwidth=0.85)
ax.set_title("Distribution of Best-Role Skill Scores", fontweight="bold")
ax.set_xlabel("Skill Score (%)")
ax.set_ylabel("Number of Freelancers")
ax.axvline(df_fl["best_score"].mean(), color="#C44E52", linestyle="--",
           label=f"Mean: {df_fl['best_score'].mean():.1f}%")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Companies & Education

In [ ]:
# ── Companies with the most freelancers ─────────────────────────────────────
company_rows = run("""
    MATCH (f:Freelancer)-[:WORKED_AT]->(c:Company)
    RETURN c.name AS company, count(DISTINCT f) AS freelancer_count
    ORDER BY freelancer_count DESC
    LIMIT 20
""")

df_companies = pd.DataFrame(company_rows)
print(f"🏢 Total unique companies: {run('MATCH (c:Company) RETURN count(c) AS n')[0]['n']}")
df_companies

In [ ]:
# ── Institutions ─────────────────────────────────────────────────────────────
inst_rows = run("""
    MATCH (f:Freelancer)-[:STUDIED_AT]->(i:Institution)
    RETURN i.name AS institution, count(DISTINCT f) AS freelancer_count
    ORDER BY freelancer_count DESC
""")

df_inst = pd.DataFrame(inst_rows)
print(f"🎓 Total unique institutions: {len(df_inst)}")
df_inst

In [ ]:
# ── Projects ─────────────────────────────────────────────────────────────────
proj_count = run("MATCH (p:Project) RETURN count(p) AS n")[0]["n"]
proj_tech_rows = run("""
    MATCH (p:Project)-[:USED_TECH]->(s:Skill)
    RETURN s.name AS tech, count(DISTINCT p) AS project_count
    ORDER BY project_count DESC
    LIMIT 15
""")

df_proj_tech = pd.DataFrame(proj_tech_rows)
print(f"📁 Total projects in graph: {proj_count}")
print("\nMost-used technologies in projects (top 15):")
df_proj_tech

---
## 7 · Freelancer Similarity (Shared Skills)

In [ ]:
# ── Shared-skill count for every pair of freelancers ────────────────────────
pair_rows = run("""
    MATCH (f1:Freelancer)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(f2:Freelancer)
    WHERE id(f1) < id(f2)
    RETURN f1.name AS freelancer_1,
           f2.name AS freelancer_2,
           count(DISTINCT s) AS shared_skills
    ORDER BY shared_skills DESC
""")

df_pairs = pd.DataFrame(pair_rows)
print(f"👥 Freelancer pairs with at least 1 shared skill: {len(df_pairs)}")
print("\nTop 15 most similar pairs:")
df_pairs.head(15)

In [ ]:
# ── Similarity heatmap ───────────────────────────────────────────────────────
names = df_fl["name"].tolist()
sim_matrix = pd.DataFrame(0, index=names, columns=names, dtype=float)

for _, row in df_pairs.iterrows():
    f1, f2, n = row["freelancer_1"], row["freelancer_2"], row["shared_skills"]
    if f1 in sim_matrix.index and f2 in sim_matrix.columns:
        sim_matrix.loc[f1, f2] = n
        sim_matrix.loc[f2, f1] = n

# Abbreviate names for readability
short = {n: n.split()[0] + " " + n.split()[-1] if len(n.split()) > 1 else n for n in names}
sim_matrix.index   = [short[n] for n in sim_matrix.index]
sim_matrix.columns = [short[n] for n in sim_matrix.columns]

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    sim_matrix, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Shared Skills"}
)
ax.set_title("Freelancer Pairwise Similarity (Shared Skills Count)", fontweight="bold", fontsize=13)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Average similarity per freelancer ───────────────────────────────────────
avg_sim = sim_matrix.replace(0, float("nan")).mean(axis=1).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
avg_sim.sort_values(ascending=True).plot(kind="barh", ax=ax, color=ACCENT, edgecolor="white")
ax.set_title("Average Shared Skills with Other Freelancers", fontweight="bold")
ax.set_xlabel("Avg. Shared Skill Count")
for bar in ax.patches:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 8 · Shared Nodes Similarity (All Relationship Types)

In [ ]:
# ── Counts shared neighbours of ANY type ─────────────────────────────────────
# (skills, domains, companies, institutions — anything connected)
shared_rows = run("""
    MATCH (f1:Freelancer)-[]-(shared)-[]-(f2:Freelancer)
    WHERE id(f1) < id(f2)
      AND NOT (shared:Freelancer)
    RETURN f1.name AS freelancer_1,
           f2.name AS freelancer_2,
           count(DISTINCT shared) AS shared_nodes
    ORDER BY shared_nodes DESC
    LIMIT 20
""")

df_shared = pd.DataFrame(shared_rows)
print("Most connected pairs (any shared entity):")
df_shared.head(15)

---
## 9 · Graph Density & Connectivity

In [ ]:
# ── Degree distribution — how many relationships does each freelancer have? ──
degree_rows = run("""
    MATCH (f:Freelancer)
    OPTIONAL MATCH (f)-[r]-()
    RETURN f.name AS name, count(r) AS degree
    ORDER BY degree DESC
""")

df_degree = pd.DataFrame(degree_rows)
print("Relationship degree per freelancer:")
print(df_degree.to_string(index=False))
print(f"\nMean degree : {df_degree['degree'].mean():.1f}")
print(f"Max  degree : {df_degree['degree'].max()}")
print(f"Min  degree : {df_degree['degree'].min()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_degree.sort_values("degree", ascending=True).set_index("name")["degree"].plot(
    kind="barh", ax=axes[0], color="#DD8452", edgecolor="white"
)
axes[0].set_title("Total Relationship Degree per Freelancer", fontweight="bold")
axes[0].set_xlabel("Degree (# edges)")
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f"{int(bar.get_width())}", va="center", fontsize=8)

axes[1].hist(df_degree["degree"], bins=8, color="#DD8452", edgecolor="white", rwidth=0.85)
axes[1].set_title("Degree Distribution", fontweight="bold")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("# Freelancers")

plt.tight_layout()
plt.show()

In [ ]:
# ── Graph density summary ────────────────────────────────────────────────────
total_nodes = df_nodes["count"].sum()
total_rels  = df_rels["count"].sum()
n_fl        = len(df_fl)
n_skills    = len(df_skills)
n_domains   = len(df_domains)

print("=" * 50)
print("  GRAPH DENSITY SUMMARY")
print("=" * 50)
print(f"  Total nodes           : {total_nodes:,}")
print(f"  Total relationships   : {total_rels:,}")
print(f"  Freelancers           : {n_fl}")
print(f"  Unique skills         : {n_skills}")
print(f"  Unique domains        : {n_domains}")
print(f"  Avg skills/freelancer : {df_fl['n_skills'].mean():.1f}")
print(f"  Avg domains/freelancer: {df_fl['n_domains'].mean():.1f}")
print(f"  Avg projects/freelancer:{df_fl['n_projects'].mean():.1f}")
print(f"  Avg degree/freelancer : {df_degree['degree'].mean():.1f}")
print(f"  Avg experience (yrs)  : {df_fl['years_exp'].mean():.2f}")
print("=" * 50)

---
## 10 · Cleanup

In [ ]:
driver.close()
print("✅ Neo4j connection closed.")